In [0]:
df = spark.table("gold_nba_shot_quality_features")

display(df.limit(5))

In [0]:
spark.sql("""
SELECT 
  COUNT(*) AS rows,
  AVG(label) AS make_rate,
  AVG(shot_dist) AS avg_shot_dist,
  AVG(close_def_dist) AS avg_close_def_dist,
  AVG(is_three_pointer) AS three_point_rate,
  AVG(is_late_clock) AS late_clock_rate,
  AVG(is_very_tight_defense) AS very_tight_defense_rate,
  AVG(is_tight_defense) AS tight_defense_rate,
  AVG(is_open_shot) AS open_shot_rate,
  AVG(is_wide_open_shot) AS wide_open_shot_rate,
  AVG(is_close_shot) AS close_shot_rate,
  AVG(is_mid_range) AS mid_range_or_long_two_rate,
  AVG(is_deep_three) AS deep_three_rate
FROM gold_nba_shot_quality_features
""").show()

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [0]:
input_cols = [
    "period",
    "shot_clock",
    "dribbles",
    "touch_time",
    "shot_dist",
    "pts_type",
    "close_def_dist",
    "is_three_pointer",
    "is_late_clock",
    "is_very_tight_defense",
    "is_tight_defense",
    "is_open_shot",
    "is_wide_open_shot",
    "is_close_shot",
    "is_mid_range",
    "is_deep_three"
]

In [0]:
assembler = VectorAssembler(
    inputCols=input_cols,
    outputCol="features",
    handleInvalid="skip"
)

model_df = assembler.transform(df).select("features", "label")

display(model_df.limit(5))

In [0]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=20
)

lr_model = lr.fit(train_df)

lr_predictions = lr_model.transform(test_df)

display(lr_predictions.limit(10))

In [0]:
binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

lr_auc = binary_evaluator.evaluate(lr_predictions)
lr_accuracy = accuracy_evaluator.evaluate(lr_predictions)

print(f"Logistic Regression AUC: {lr_auc}")
print(f"Logistic Regression Accuracy: {lr_accuracy}")

In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    numTrees=50,
    maxDepth=5,
    seed=42
)

rf_model = rf.fit(train_df)

rf_predictions = rf_model.transform(test_df)

display(rf_predictions.limit(10))

In [0]:
rf_auc = binary_evaluator.evaluate(rf_predictions)
rf_accuracy = accuracy_evaluator.evaluate(rf_predictions)

print(f"Random Forest AUC: {rf_auc}")
print(f"Random Forest Accuracy: {rf_accuracy}")

In [0]:
from pyspark.sql import Row

metrics_df = spark.createDataFrame([
    Row(model_name="logistic_regression", metric_name="auc", metric_value=float(lr_auc)),
    Row(model_name="logistic_regression", metric_name="accuracy", metric_value=float(lr_accuracy)),
    Row(model_name="random_forest", metric_name="auc", metric_value=float(rf_auc)),
    Row(model_name="random_forest", metric_name="accuracy", metric_value=float(rf_accuracy))
])

metrics_df.write.mode("overwrite").saveAsTable("gold_nba_shot_model_metrics")

display(metrics_df)

In [0]:
import pandas as pd

feature_importance = rf_model.featureImportances.toArray().tolist()

importance_pdf = pd.DataFrame({
    "feature": input_cols,
    "importance": feature_importance
}).sort_values("importance", ascending=False)

feature_importance_df = spark.createDataFrame(importance_pdf)

feature_importance_df.write.mode("overwrite").saveAsTable("gold_nba_shot_feature_importance")

display(feature_importance_df)